In [2]:
import pickle
import pandas as pd

### Required Candidates files

In [64]:
path = "candidates.pkl"
with open(path, 'rb') as file:
    df_candidates = pickle.load(file)

In [65]:
df_candidates.columns

Index(['candidate_id', 'text', 'skills', 'timestamp', 'embedding',
       'experience', 'location', 'salary_expectation', 'origin'],
      dtype='object')

In [66]:
df_candidates.isnull().sum()

candidate_id          0
text                  0
skills                0
timestamp             0
embedding             0
experience            0
location              0
salary_expectation    0
origin                0
dtype: int64

In [67]:
df_candidates.head()

,candidate_id,text,skills,timestamp,embedding,experience,location,salary_expectation,origin
0,candidate_0,"13 years of exp || Solidity, C#, JavaScript ||...","[javascript, solidity, sql, rust, c#, react, j...",1701302400,"[-0.072871156, -0.003732475, -0.017213404, -0....",junior,location_86,90-120k,indeed
1,candidate_1,1c Developer \nWorked on a mobile application ...,[r],1703894400,"[-0.087375894, -0.031941064, -0.0037780453, 0....",senior,location_0,120-150k,indeed
2,candidate_2,1C developer \n1 am an 1C developer. I deploye...,"[rest, api, r]",1689120000,"[-0.032614697, -0.04999506, -0.024087394, -0.0...",lead,location_26,90-120k,linkedin
3,candidate_3,1C Developer \nPerfect knowledge of 1C:Enterp...,"[sql, postgresql, docker, java, git, rest, r, ...",1693526400,"[-0.05305553, -0.066774346, -0.04145598, -1.94...",entry,location_46,120-150k,indeed
4,candidate_4,#1 Customer Support Specialist thousands of re...,[r],1703462400,"[-0.0867067, -0.08233174, 0.09725701, 0.005730...",senior,location_37,50-70k,linkedin


#### preprocess the text

In [17]:
import re

PARENTHESIS = re.compile(r"\([^)]*\)")
YEARS_EXP = re.compile(r"\b\d+\s+years?\b")

NORMALIZATION = {
    "react.js": "react",
    "node.js": "nodejs",
    ".net": "dotnet",
    "c#": "csharp",
    "c sharp": "csharp",
    "web3.js": "web3js",
    "ether.js": "ethers",
}

def preprocess_text(text: str) -> str:
    if not isinstance(text, str):
        return ""

    text = text.lower()

    # normalize separators
    text = text.replace("||", " ").replace("/", " ")

    # remove parentheses content
    text = PARENTHESIS.sub(" ", text)

    # remove years mentions
    text = YEARS_EXP.sub(" ", text)

    # normalize tech tokens
    for k, v in NORMALIZATION.items():
        text = text.replace(k, v)

    # collapse whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text
df_candidates["text"] = df_candidates["text"].apply(preprocess_text)


In [68]:
df_candidates.head()

,candidate_id,text,skills,timestamp,embedding,experience,location,salary_expectation,origin
0,candidate_0,"13 years of exp || Solidity, C#, JavaScript ||...","[javascript, solidity, sql, rust, c#, react, j...",1701302400,"[-0.072871156, -0.003732475, -0.017213404, -0....",junior,location_86,90-120k,indeed
1,candidate_1,1c Developer \nWorked on a mobile application ...,[r],1703894400,"[-0.087375894, -0.031941064, -0.0037780453, 0....",senior,location_0,120-150k,indeed
2,candidate_2,1C developer \n1 am an 1C developer. I deploye...,"[rest, api, r]",1689120000,"[-0.032614697, -0.04999506, -0.024087394, -0.0...",lead,location_26,90-120k,linkedin
3,candidate_3,1C Developer \nPerfect knowledge of 1C:Enterp...,"[sql, postgresql, docker, java, git, rest, r, ...",1693526400,"[-0.05305553, -0.066774346, -0.04145598, -1.94...",entry,location_46,120-150k,indeed
4,candidate_4,#1 Customer Support Specialist thousands of re...,[r],1703462400,"[-0.0867067, -0.08233174, 0.09725701, 0.005730...",senior,location_37,50-70k,linkedin


### Creating candidates.csv
#### contains candidate_id, name, description, timestamp

In [91]:
candidates = df_candidates[[
    "candidate_id",
    "text",
    "timestamp"
]].copy()

candidates.columns = [
    "candidate_ID",
    "description",
    "timestamp"
]

# use candidate_id as name (required for joins later)
candidates["name"] = candidates["candidate_ID"]

# reorder columns (optional but clean)
candidates = candidates[
    ["candidate_ID", "name", "description", "timestamp"]
]

candidates.to_csv("data/candidates.csv", index=False)


### Creating candidates_preprocessing.csv
#### contains extra related features

In [ ]:
candidate_pre = pd.DataFrame()

# required identifier
candidate_pre["Name"] = df_candidates["candidate_id"]

# experience (already normalized)
candidate_pre["experience"] = df_candidates["experience"]

# origin
candidate_pre["origin"] = df_candidates["origin"]

# salary (placeholder, safe)
candidate_pre["salary_current"] = "0.0"

# daily rate placeholder
candidate_pre["daily_rate"] = 0.0

# zip / location placeholder
candidate_pre["zip"] = -1

# optional fields (safe defaults)
candidate_pre["contract"] = ""
candidate_pre["category"] = ""
candidate_pre["recruiter"] = ""
candidate_pre["Date Added"] = df_candidates["timestamp"] * 1000

# final cleanup
candidate_pre = candidate_pre.fillna(-1)

candidate_pre.to_csv("data/candidate_preprocessing.csv", index=False)


In [61]:
candidate_pre["salary_current"].apply(type).value_counts()

salary_current
<class 'int'>    210250
Name: count, dtype: int64

### Creating candidates_skills.csv
#### mapping each candidates to their skills

In [ ]:
rows = []

for _, row in df_candidates.iterrows():
    candidate_name = row["candidate_id"]
    for skill in row["skills"]:   # list of skills
        rows.append([candidate_name, skill.lower()])

candidate_skills = pd.DataFrame(
    rows,
    columns=["name", "skillUri"]
).drop_duplicates()

candidate_skills.to_csv("data/candidate_skills.csv", index=False)


### Required Jobs files

In [71]:
path = "jobs.pkl"
with open(path, 'rb') as file:
    df_jobs = pickle.load(file)

In [72]:
df_jobs.head()

,job_id,text,skills,company,timestamp,embedding,experience_required,location,salary_range,contract_type,category
0,job_0,10 + Blockchain Nodes / Masternodes to set up ...,"[node, unity, blockchain, r]",MyCointainer,1698364800,"[-0.03756034, -0.07931591, 0.06068252, -0.0445...",senior,location_92,120-150k,contract,designer
1,job_1,10 .NET Developers (Middle and Senior level) G...,"[api, sql, c#, react, rest, r, angular, micros...",TechScout.tech,1699488000,"[-0.09647457, -0.075682655, -0.013227892, -0.0...",principal,location_26,150k+,temporary,designer
2,job_2,"10X Engineer (co-founder, #4 employee, USD 11-...","[sql, flask, php, c#, react, java, rest, types...",Innoteka,1703376000,"[-0.08963827, -0.0420551, 0.020479323, -0.0848...",senior,location_57,120-150k,freelance,fullstack
3,job_3,"16 - Amazon Brand Manager Currently, TCM expan...","[unity, rest, r]",FirstFive,1701820800,"[0.010399047, -0.019235075, -0.0039642523, 0.0...",lead,location_27,30-50k,freelance,backend
4,job_4,"16 - Amazon Brand Manager Hello,\r\nWe, MIMIRB...",[r],MimirB2B,1700956800,"[-0.0056338236, -0.022090638, -0.033225723, 0....",lead,location_41,30-50k,contract,product_manager


#### preprocessing text

In [25]:
df_jobs["text"] = df_jobs["text"].apply(preprocess_text)

In [26]:
df_jobs["location"].nunique()

100

### Creating jobs.csv

In [ ]:
jobs = df_jobs[[
    "job_id",
    "text",
    "company",
    "timestamp"
]].copy()

jobs.columns = [
    "job_ID",
    "description",
    "JobOrder.clientCorporation",
    "timestamp"
]

# Use job_id as Job identifier (required for joins)
jobs["Job"] = jobs["job_ID"]

# Reorder columns (clean + expected)
jobs = jobs[
    ["job_ID", "Job", "JobOrder.clientCorporation", "description", "timestamp"]
]

jobs.to_csv("data/jobs.csv", index=False)


### Creating job_skills.csv
#### mapping jobs to skills required

In [63]:
rows = []

for _, row in jobs.iterrows():
    job_name = row["job_id"]
    for skill in row["skills"]:   # list of skills
        rows.append([job_name, skill.lower()])

jobs_skills = pd.DataFrame(
    rows,
    columns=["Job", "skillUri"]
).drop_duplicates()

jobs_skills.to_csv("data/jobs_skills.csv", index=False)


In [30]:
df = pd.read_csv("edges.csv")
df.columns

Index(['source_type', 'source_id', 'relation', 'target_type', 'target_id'], dtype='object')

In [31]:
df["target_type"].unique()

array(['skill', 'skill_concept', 'experience', 'location', 'salary',
       'origin', 'company', 'contract_type', 'job_category', 'candidate',
       'job', 'temporal'], dtype=object)

In [32]:
df["relation"].unique()

array(['hasSkill', 'belongsTo', 'hasExperience', 'atLocation',
       'expectsSalary', 'fromOrigin', 'requiresSkill', 'atCompany',
       'requiresExperience', 'offersSalary', 'hasContract', 'inCategory',
       'hasCandidate', 'forJob', 'atTime', 'activeAt'], dtype=object)

In [33]:
df[(df["relation"] == "belongsTo")]

,source_type,source_id,relation,target_type,target_id
1,skill,javascript,belongsTo,skill_concept,programming
3,skill,solidity,belongsTo,skill_concept,web3
5,skill,sql,belongsTo,skill_concept,database
7,skill,rust,belongsTo,skill_concept,programming
9,skill,c#,belongsTo,skill_concept,programming
...,...,...,...,...,...
2188986,skill,sql,belongsTo,skill_concept,database
2188988,skill,java,belongsTo,skill_concept,programming
2189043,skill,rust,belongsTo,skill_concept,programming
2189052,skill,aws,belongsTo,skill_concept,cloud


In [32]:
unique_skills = set()

for skills in jobs["skills"]:
    for skill in skills:
        unique_skills.add(skill.lower().strip())

unique_skills = sorted(unique_skills)

In [ ]:
unique_skills

['agile',
 'android',
 'angular',
 'ansible',
 'api',
 'aws',
 'blockchain',
 'c#',
 'c++',
 'ci/cd',
 'computer vision',
 'deep learning',
 'devops',
 'django',
 'docker',
 'elasticsearch',
 'express',
 'fastapi',
 'flask',
 'flutter',
 'git',
 'gitlab',
 'golang',
 'graphql',
 'hadoop',
 'ios',
 'java',
 'javascript',
 'jenkins',
 'jira',
 'kafka',
 'kotlin',
 'kubernetes',
 'linux',
 'machine learning',
 'matlab',
 'microservices',
 'mongodb',
 'nestjs',
 'nlp',
 'node',
 'numpy',
 'pandas',
 'php',
 'postgresql',
 'python',
 'pytorch',
 'r',
 'react',
 'reactnative',
 'redis',
 'rest',
 'ruby',
 'rust',
 'scala',
 'scikit-learn',
 'scrum',
 'solidity',
 'spark',
 'spring',
 'sql',
 'swift',
 'tensorflow',
 'terraform',
 'typescript',
 'unity',
 'unreal',
 'vue']

### Creating hierarchy.csv

In [34]:
hierarchy = df[df["relation"] == "belongsTo"][
    ["target_id", "source_id"]
].drop_duplicates()

hierarchy.columns = ["broaderUri", "conceptUri"]

# add labels (simple version)
hierarchy["preferredLabel_child"] = hierarchy["conceptUri"]
hierarchy["preferredLabel_parent"] = hierarchy["broaderUri"]

# optional placeholders to match old schema
hierarchy["altLabels_child"] = ""
hierarchy["altLabels_parent"] = ""


In [ ]:
hierarchy.to_csv("data/hierarchy.csv", index=False)

In [38]:
df_interactions = pd.read_csv("train_interactions.csv")

In [39]:
df_interactions.head()

,shortlist_id,candidate_id,job_id,timestamp,date,months_since_start
0,shortlist_20183,candidate_55877,job_17817,1704067200,2024-01-01,0
1,shortlist_22215,candidate_46906,job_67270,1704067200,2024-01-01,0
2,shortlist_61263,candidate_77000,job_40131,1704067200,2024-01-01,0
3,shortlist_15244,candidate_157183,job_103513,1704067200,2024-01-01,0
4,shortlist_15245,candidate_18306,job_95896,1704067200,2024-01-01,0


In [40]:
df_interactions.isna().sum()

shortlist_id          0
candidate_id          0
job_id                0
timestamp             0
date                  0
months_since_start    0
dtype: int64

### Creating job_candidates.csv

In [ ]:
job_candidates = df_interactions[[
    "candidate_id",
    "job_id",
    "date",
    "timestamp"
]].copy()

job_candidates.columns = [
    "candidate_ID",
    "job_ID",
    "Date Added_caller",
    "timestamp"
]

# identifiers used for joins later
job_candidates["name"] = job_candidates["candidate_ID"]
job_candidates["Job"] = job_candidates["job_ID"]

# ensure datetime (important)
job_candidates["Date Added_caller"] = (
    pd.to_datetime(job_candidates["timestamp"], unit="s")
      .dt.strftime("%Y-%m-%d %H:%M:%S")
)
# reorder columns (clean + expected)
job_candidates = job_candidates[
    ["candidate_ID", "job_ID", "name", "Job", "Date Added_caller", "timestamp"]
]

job_candidates.to_csv("data/job_candidates.csv", index=False)


### Creating job_preprocessing.csv

In [ ]:
import pandas as pd
from datetime import datetime
job_pre = pd.DataFrame()

# identifiers
job_pre["title"] = df_jobs["job_id"]

# convert timestamp → DD/MM/YYYY (required by code)
job_pre["dateAdded"] = pd.to_datetime(
    df_jobs["timestamp"], unit="s"
).dt.strftime("%d/%m/%Y")

# experience requirement
job_pre["customText6"] = df_jobs["experience_required"]

# contract type
job_pre["employmentType"] = df_jobs["contract_type"]

# categories
job_pre["categories"] = df_jobs["category"]

# placeholders / safe defaults
job_pre["salary"] = 0
job_pre["payRate"] = 0
job_pre["zip"] = -1
job_pre["owner"] = ""
job_pre["latitude"] = 0.0
job_pre["longitude"] = 0.0

# final ordering (clean, optional)
job_pre = job_pre[
    ["title", "dateAdded", "salary", "payRate", "zip",
     "owner", "categories", "employmentType", "customText6"]
]

job_pre.to_csv("data/job_preprocessing.csv", index=False)


### Creating cities.csv

In [52]:
cities =  [[-1, 0.0, 0.0] for _ in range(100)]

In [53]:
df_cities = pd.DataFrame(cities, columns= ["zip_code", "latitude", "longitude"])

In [54]:
df_cities.head()

,zip_code,latitude,longitude
0,-1,0.0,0.0
1,-1,0.0,0.0
2,-1,0.0,0.0
3,-1,0.0,0.0
4,-1,0.0,0.0


In [ ]:
df_cities.to_csv("data/cities.csv", index=False)

# NOTE: 
### Few fields have been filled with dummy values: example- zip, longitude, latitude, salary, etc.